In [1]:
import pandas as pd
from omegaconf import DictConfig
import omegaconf
import psycopg2

In [2]:
cfg = omegaconf.OmegaConf.load("../config.yaml")

In [3]:
conn = psycopg2.connect(**cfg.db)

In [4]:
def prepare_table(df, column_name = 'search_month', value_column = 'search_count'):
    df[column_name] = df[column_name].astype(str)
    df = df.pivot_table(index='query_id', columns=column_name, values=value_column)
    df.fillna(0, inplace=True)
    df = df.astype(int)
    return df

In [5]:
def make_snapshots(df):
    df = df.copy()
    df["pad-pre"] = df["2025-01-01"] + df["2025-02-01"]
    df["snapshot-1"] = df["2025-03-01"] + df["2025-04-01"] + df["2025-05-01"]
    df["snapshot-2"] = df["2025-06-01"] + df["2025-07-01"] + df["2025-08-01"]
    df["snapshot-3"] = df["2025-09-01"] + df["2025-10-01"] + df["2025-11-01"]
    df["pad-past"] = df["2025-12-01"]

    df = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past"]]
    return df

In [6]:
query = """
SELECT query_monthly_stats.query_id, query_monthly_stats.search_month, query_monthly_stats.search_count
FROM query_monthly_stats
JOIN public.queries q on query_monthly_stats.query_id = q.query_id
WHERE q.is_structured is FALSE AND q.is_boolean is FALSE and q.has_doi is FALSE and q.was_clicked is TRUE and num_words >= 2 and num_words <= 10;
"""

df = pd.read_sql(query, conn)

/tmp/ipykernel_2554838/58730802.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [7]:
df = prepare_table(df)
df = make_snapshots(df)

## normalize

In [8]:
import numpy as np

In [9]:
def normalize_unequal_snapshots(row):
    p1, s1, s2, s3, p2 = row["pad-pre"], row["snapshot-1"], row["snapshot-2"], row["snapshot-3"], row["pad-past"]
    durations = np.array([59, 92, 92, 91, 5])     
    rates = np.array([p1, s1, s2, s3, p2]) / durations

    rates += 1e-9  # avoide 0 division issues
    
    # L1 Normalization
    distribution = rates / np.sum(rates)
    
    return distribution

In [10]:
df[["n-pad-pre", "n-snapshot-1", "n-snapshot-2", "n-snapshot-3", "n-pad-past"]] = df.apply(normalize_unequal_snapshots, axis=1, result_type='expand')

In [11]:
df["sum"] = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past"]].sum(axis=1)

In [12]:
df.sort_values("sum", ascending=False, inplace=True)

In [13]:
df

search_month,pad-pre,snapshot-1,snapshot-2,snapshot-3,pad-past,n-pad-pre,n-snapshot-1,n-snapshot-2,n-snapshot-3,n-pad-past,sum
query_id,,,,,,,,,,,
6085264,0,0,22,10311,0,8.806940e-12,8.806940e-12,0.002106,9.978940e-01,8.806940e-12,10333
8033880,3764,1857,1724,1797,100,4.477968e-01,1.416796e-01,0.131532,1.386086e-01,1.403826e-01,9242
13814606,51,4771,3859,11,0,9.119212e-03,5.470925e-01,0.442513,1.275236e-03,1.054968e-11,8692
14048082,53,4762,3853,0,0,9.501891e-03,5.475046e-01,0.442994,1.057758e-11,1.057758e-11,8668
14262547,63,4768,3831,0,0,1.129523e-02,5.482201e-01,0.440485,1.057807e-11,1.057807e-11,8662
...,...,...,...,...,...,...,...,...,...,...,...
16666254,0,0,0,0,0,2.000000e-01,2.000000e-01,0.200000,2.000000e-01,2.000000e-01,0
16667095,0,0,0,0,0,2.000000e-01,2.000000e-01,0.200000,2.000000e-01,2.000000e-01,0
16667028,0,0,0,0,0,2.000000e-01,2.000000e-01,0.200000,2.000000e-01,2.000000e-01,0


In [14]:
from scipy.special import rel_entr

In [15]:
def detect_pattern(data_matrix, pattern):
    kl_divs = np.sum(rel_entr(pattern, data_matrix), axis=1)
    similarity = 1 / (1 + np.abs(kl_divs)) # convert to similarity score
    return similarity

In [16]:
def detect_slope(data_matrix):
    x = np.array([0, 1, 2, 3, 4])
    x_mean = np.mean(x)
    
    # Pre-calculate denominator for slope: sum((x - x_mean)^2)
    x_ss = np.sum((x - x_mean)**2) 
    
    # Matrix-wise means for each row
    y_means = np.mean(data_matrix, axis=1, keepdims=True)
    
    # Calculate slopes (m) for all rows at once
    # m = sum((x - x_mean) * (y - y_mean)) / sum((x - x_mean)^2)
    slopes = np.dot(data_matrix - y_means.flatten()[:, None], (x - x_mean)) / x_ss
    
    # Calculate intercepts (c)
    intercepts = y_means.flatten() - slopes * x_mean
    
    # Calculate Residuals
    # y_fit = m * x + c
    y_fit = slopes[:, None] * x + intercepts[:, None]
    residuals = np.sum((data_matrix - y_fit)**2, axis=1)
    
    # Total Sum of Squares (SS_tot)
    # Sum of squared deviations from the mean
    ss_tot = np.sum((data_matrix - y_means)**2, axis=1)
    
    # Calculate R-squared
    # We use np.where to handle cases with no variance (perfectly flat lines)
    r_squared = np.where(ss_tot > 1e-12, 1 - (residuals / ss_tot), 0.0)
    
    return slopes, residuals, r_squared



In [17]:
data_matrix = df[["n-pad-pre", "n-snapshot-1", "n-snapshot-2", "n-snapshot-3", "n-pad-past"]].values

In [18]:
spike_1 = [1e-9, 0.89, 1e-9, 1e-9, 1e-9]
df['spike_1'] = detect_pattern(data_matrix, spike_1)

spike_2 = [1e-9, 1e-9, 0.89, 1e-9, 1e-9]
df['spike_2'] = detect_pattern(data_matrix, spike_2)

spike_3 = [1e-9, 1e-9, 1e-9, 0.89, 1e-9]
df['spike_3'] = detect_pattern(data_matrix, spike_3)

emerging_1 = [1e-9, 0.89, 0.89, 0.89, 0.89]
df['emerging_1'] = detect_pattern(data_matrix, emerging_1)

emerging_2 = [1e-9, 1e-9, 0.89, 0.89, 0.89]
df['emerging_2'] = detect_pattern(data_matrix, emerging_2)

emerging_3 = [1e-9, 1e-9, 1e-9, 0.89, 0.89]
df['emerging_3'] = detect_pattern(data_matrix, emerging_3)

vanishing_1 = [0.89, 1e-9, 1e-9, 1e-9, 1e-9]
df['vanishing_1'] = detect_pattern(data_matrix, vanishing_1)

vanishing_2 = [0.89, 0.89, 1e-9, 1e-9, 1e-9]
df['vanishing_2'] = detect_pattern(data_matrix, vanishing_2)

vanishing_3 = [0.89, 0.89, 0.89, 1e-9, 1e-9]
df['vanishing_3'] = detect_pattern(data_matrix, vanishing_3)


df['slope'], df['residuals'], df['r_squared'] = detect_slope(data_matrix)

/tmp/ipykernel_2554838/4110517515.py:29: RuntimeWarning: invalid value encountered in divide
  r_squared = np.where(ss_tot > 1e-12, 1 - (residuals / ss_tot), 0.0)


In [19]:
df

search_month,pad-pre,snapshot-1,snapshot-2,snapshot-3,pad-past,n-pad-pre,n-snapshot-1,n-snapshot-2,n-snapshot-3,n-pad-past,...,spike_3,emerging_1,emerging_2,emerging_3,vanishing_1,vanishing_2,vanishing_3,slope,residuals,r_squared
query_id,,,,,,,,,,,,,,,,,,,,,
6085264,0,0,22,10311,0,8.806940e-12,8.806940e-12,0.002106,9.978940e-01,8.806940e-12,...,0.907574,0.019462,0.034685,0.042644,0.042460,0.021690,0.019423,0.099789,0.696218,0.125131
8033880,3764,1857,1724,1797,100,4.477968e-01,1.416796e-01,0.131532,1.386086e-01,1.403826e-01,...,0.376646,0.130961,0.166657,0.232628,0.620607,0.307992,0.202082,-0.061790,0.038635,0.497035
13814606,51,4771,3859,11,0,9.119212e-03,5.470925e-01,0.442513,1.275236e-03,1.054968e-11,...,0.146460,0.033032,0.033511,0.034225,0.196969,0.181487,0.163081,-0.056406,0.263397,0.107773
14048082,53,4762,3853,0,0,9.501891e-03,5.475046e-01,0.442994,1.057758e-11,1.057758e-11,...,0.042756,0.021354,0.021553,0.021845,0.198399,0.182723,0.164104,-0.056651,0.264002,0.108388
14262547,63,4768,3831,0,0,1.129523e-02,5.482201e-01,0.440485,1.057807e-11,1.057807e-11,...,0.042756,0.021352,0.021550,0.021845,0.204646,0.188050,0.168245,-0.057081,0.262117,0.110562
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16666254,0,0,0,0,0,2.000000e-01,2.000000e-01,0.200000,2.000000e-01,2.000000e-01,...,0.429427,0.158360,0.200559,0.273421,0.429427,0.273421,0.200559,0.000000,0.000000,0.000000
16667095,0,0,0,0,0,2.000000e-01,2.000000e-01,0.200000,2.000000e-01,2.000000e-01,...,0.429427,0.158360,0.200559,0.273421,0.429427,0.273421,0.200559,0.000000,0.000000,0.000000
16667028,0,0,0,0,0,2.000000e-01,2.000000e-01,0.200000,2.000000e-01,2.000000e-01,...,0.429427,0.158360,0.200559,0.273421,0.429427,0.273421,0.200559,0.000000,0.000000,0.000000


# Click Support

In [20]:
query = """
SELECT query_monthly_click_stats.query_id, query_monthly_click_stats.click_month, query_monthly_click_stats.total_clicks
FROM query_monthly_click_stats
JOIN public.queries q on query_monthly_click_stats.query_id = q.query_id
WHERE q.is_structured is FALSE AND q.is_boolean is FALSE and q.has_doi is FALSE and q.was_clicked is TRUE;
"""

click_df = pd.read_sql(query, conn)

/tmp/ipykernel_2554838/2871406387.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  click_df = pd.read_sql(query, conn)


In [21]:
click_df = prepare_table(click_df, column_name='click_month', value_column='total_clicks')
click_df = make_snapshots(click_df)

In [22]:
click_df = click_df.add_prefix("click-")

In [23]:
df = df.merge(click_df, left_index=True, right_index=True, how='left')

# Select candidates

In [24]:
candidates = []

### Spike

In [25]:
spike_1_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "spike_1"]].copy()
spike_1_candidates["sorter"] = spike_1_candidates["spike_1"] * np.log10(spike_1_candidates["click-snapshot-1"])
spike_1_candidates = spike_1_candidates.sort_values("sorter", ascending=False).head(100).drop(columns=["spike_1"])
spike_1_candidates["temporal-type"] = "spike-1"
spike_1_candidates["rank"] = np.arange(1, len(spike_1_candidates) + 1)
candidates.append(spike_1_candidates)

/workspaces/conf26-longeval-sci/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [26]:
spike_2_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "spike_2"]].copy()
spike_2_candidates["sorter"] = spike_2_candidates["spike_2"] * np.log10(spike_2_candidates["click-snapshot-2"])
spike_2_candidates = spike_2_candidates.sort_values("sorter", ascending=False).head(100).drop(columns=["spike_2"])
spike_2_candidates["temporal-type"] = "spike-2"
spike_2_candidates["rank"] = np.arange(1, len(spike_2_candidates) + 1)
candidates.append(spike_2_candidates)

/workspaces/conf26-longeval-sci/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [27]:
spike_3_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "spike_3"]].copy()
spike_3_candidates["sorter"] = spike_3_candidates["spike_3"] * np.log10(spike_3_candidates["click-snapshot-3"])
spike_3_candidates = spike_3_candidates.sort_values("sorter", ascending=False).head(100).drop(columns=["spike_3"])
spike_3_candidates["temporal-type"] = "spike-3"
spike_3_candidates["rank"] = np.arange(1, len(spike_3_candidates) + 1)
candidates.append(spike_3_candidates)

/workspaces/conf26-longeval-sci/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


### Stable

In [28]:
stable_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "slope",	"residuals"]].copy()

min_click_threshold = 30
mask = (stable_candidates["click-snapshot-1"] < min_click_threshold) | (stable_candidates["click-snapshot-2"] < min_click_threshold) | (stable_candidates["click-snapshot-3"] < min_click_threshold)
stable_candidates = stable_candidates[~mask]

# sorter
stable_candidates["sorter"] = (np.exp(-5.0 * (abs(stable_candidates["slope"]) + stable_candidates["residuals"]))) * (np.log10(stable_candidates["click-snapshot-1"] + stable_candidates["click-snapshot-2"] + stable_candidates["click-snapshot-3"] + 1))

stable_candidates["temporal-type"] = "stable"
stable_candidates = stable_candidates.sort_values("sorter", ascending=False)
stable_candidates["rank"] = np.arange(1, len(stable_candidates) + 1)
stable_candidates.drop(columns=["slope", "residuals"], inplace=True)
candidates.append(stable_candidates)

## Emerging

In [29]:
emerging_1_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "emerging_1"]].copy()

# min clicks
min_click_threshold = 30
mask = (emerging_1_candidates["click-snapshot-1"] < min_click_threshold) | (emerging_1_candidates["click-snapshot-2"] < min_click_threshold) | (emerging_1_candidates["click-snapshot-3"] < min_click_threshold)
emerging_1_candidates = emerging_1_candidates[~mask]

emerging_1_candidates["sorter"] = emerging_1_candidates["emerging_1"] * np.log10(emerging_1_candidates["click-snapshot-1"] + emerging_1_candidates["click-snapshot-2"] + emerging_1_candidates["click-snapshot-3"] + 1 )

emerging_1_candidates["temporal-type"] = "emerging-1"
emerging_1_candidates = emerging_1_candidates.sort_values("sorter", ascending=False).head(50)
emerging_1_candidates["rank"] = np.arange(1, len(emerging_1_candidates) + 1)
emerging_1_candidates.drop(columns=["emerging_1"], inplace=True)
candidates.append(emerging_1_candidates)

In [30]:
emerging_2_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "emerging_2"]].copy()

# min clicks
min_click_threshold = 30
mask = (emerging_2_candidates["click-snapshot-2"] < min_click_threshold) | (emerging_2_candidates["click-snapshot-3"] < min_click_threshold)
emerging_2_candidates = emerging_2_candidates[~mask]

emerging_2_candidates["sorter"] = emerging_2_candidates["emerging_2"] * np.log10(emerging_2_candidates["click-snapshot-2"] + emerging_2_candidates["click-snapshot-3"] + 1 )

emerging_2_candidates["temporal-type"] = "emerging-2"
emerging_2_candidates = emerging_2_candidates.sort_values("sorter", ascending=False).head(50)
emerging_2_candidates["rank"] = np.arange(1, len(emerging_2_candidates) + 1)
emerging_2_candidates.drop(columns=["emerging_2"], inplace=True)
candidates.append(emerging_2_candidates)

In [31]:
emerging_3_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "emerging_3"]].copy()

# min clicks
min_click_threshold = 30
mask = (emerging_3_candidates["click-snapshot-3"] < min_click_threshold)
emerging_3_candidates = emerging_3_candidates[~mask]

emerging_3_candidates["sorter"] = emerging_3_candidates["emerging_3"] * np.log10(emerging_3_candidates["click-snapshot-3"] + 1 )

emerging_3_candidates["temporal-type"] = "emerging-3"
emerging_3_candidates = emerging_3_candidates.sort_values("sorter", ascending=False).head(50)
emerging_3_candidates["rank"] = np.arange(1, len(emerging_3_candidates) + 1)
emerging_3_candidates.drop(columns=["emerging_3"], inplace=True)
candidates.append(emerging_3_candidates)

## Vanishing

In [32]:
vanishing_1_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "vanishing_1"]].copy()

# min clicks
min_click_threshold = 30
mask = (vanishing_1_candidates["click-snapshot-1"] < min_click_threshold)
vanishing_1_candidates = vanishing_1_candidates[~mask]

vanishing_1_candidates["sorter"] = vanishing_1_candidates["vanishing_1"] * np.log10(vanishing_1_candidates["click-snapshot-1"] + 1)

vanishing_1_candidates["temporal-type"] = "vanishing-1"
vanishing_1_candidates = vanishing_1_candidates.sort_values("sorter", ascending=False).head(50)
vanishing_1_candidates["rank"] = np.arange(1, len(vanishing_1_candidates) + 1)
vanishing_1_candidates.drop(columns=["vanishing_1"], inplace=True)
candidates.append(vanishing_1_candidates)

In [33]:
vanishing_2_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "vanishing_2"]].copy()

# min clicks
min_click_threshold = 30
mask = (vanishing_2_candidates["click-snapshot-1"] < min_click_threshold) | (vanishing_2_candidates["click-snapshot-2"] < min_click_threshold)
vanishing_2_candidates = vanishing_2_candidates[~mask]

vanishing_2_candidates["sorter"] = vanishing_2_candidates["vanishing_2"] * np.log10(vanishing_2_candidates["click-snapshot-1"] + vanishing_2_candidates["click-snapshot-2"] + 1)

vanishing_2_candidates["temporal-type"] = "vanishing-2"
vanishing_2_candidates = vanishing_2_candidates.sort_values("sorter", ascending=False).head(50)
vanishing_2_candidates["rank"] = np.arange(1, len(vanishing_2_candidates) + 1)
vanishing_2_candidates.drop(columns=["vanishing_2"], inplace=True)
candidates.append(vanishing_2_candidates)

In [34]:
vanishing_3_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "vanishing_3"]].copy()

# min clicks
min_click_threshold = 30
mask = (vanishing_3_candidates["click-snapshot-1"] < min_click_threshold) | (vanishing_3_candidates["click-snapshot-2"] < min_click_threshold) | (vanishing_3_candidates["click-snapshot-3"] < min_click_threshold)
vanishing_3_candidates = vanishing_3_candidates[~mask]

vanishing_3_candidates["sorter"] = vanishing_3_candidates["vanishing_3"] * np.log10(vanishing_3_candidates["click-snapshot-1"] + vanishing_3_candidates["click-snapshot-2"] + vanishing_3_candidates["click-snapshot-3"] + 1)

vanishing_3_candidates["temporal-type"] = "vanishing-3"
vanishing_3_candidates = vanishing_3_candidates.sort_values("sorter", ascending=False).head(50)
vanishing_3_candidates["rank"] = np.arange(1, len(vanishing_3_candidates) + 1)
vanishing_3_candidates.drop(columns=["vanishing_3"], inplace=True)
candidates.append(vanishing_3_candidates)

## Gaining

In [35]:
gaining_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "slope",	"residuals"]].copy()

min_click_threshold = 30
mask = (gaining_candidates["click-snapshot-1"] < min_click_threshold) | (gaining_candidates["click-snapshot-2"] < min_click_threshold) | (gaining_candidates["click-snapshot-3"] < min_click_threshold)
gaining_candidates = gaining_candidates[~mask]
# sorter
gaining_candidates = gaining_candidates[gaining_candidates["slope"] >0]
gaining_candidates["sorter"] = (gaining_candidates["slope"] + gaining_candidates["residuals"]) * (np.log10(gaining_candidates["click-snapshot-1"] + gaining_candidates["click-snapshot-2"] + gaining_candidates["click-snapshot-3"] + 1))

gaining_candidates["temporal-type"] = "gaining"
gaining_candidates = gaining_candidates.sort_values("sorter", ascending=False)
gaining_candidates["rank"] = np.arange(1, len(gaining_candidates) + 1)
gaining_candidates.drop(columns=["slope", "residuals"], inplace=True)
candidates.append(gaining_candidates)

In [36]:
loosing_candidates = df[["pad-pre", "snapshot-1", "snapshot-2", "snapshot-3", "pad-past", "click-pad-pre", "click-snapshot-1", "click-snapshot-2", "click-snapshot-3", "click-pad-past", "slope",	"residuals", "r_squared"]].copy()

min_click_threshold = 30
mask = (loosing_candidates["click-snapshot-1"] < min_click_threshold) | (loosing_candidates["click-snapshot-2"] < min_click_threshold) | (loosing_candidates["click-snapshot-3"] < min_click_threshold)
loosing_candidates = loosing_candidates[~mask]


# sorter
loosing_candidates = loosing_candidates[loosing_candidates["slope"] >0]
# loosing_candidates["sorter"] = (
#     abs(loosing_candidates["slope"]) * loosing_candidates["r_squared"]
# )
loosing_candidates["sorter"] = (
    abs(loosing_candidates["slope"]) * loosing_candidates["r_squared"]
) * (
    np.log10(loosing_candidates["click-snapshot-1"] + 
             loosing_candidates["click-snapshot-2"] + 
             loosing_candidates["click-snapshot-3"] + 1)
)

loosing_candidates["temporal-type"] = "loosing"
loosing_candidates = loosing_candidates.sort_values("sorter", ascending=False)
loosing_candidates["rank"] = np.arange(1, len(loosing_candidates) + 1)
# loosing_candidates.drop(columns=["slope", "residuals"], inplace=True)
# candidates.append(loosing_candidates)

In [37]:
loosing_candidates.sort_values("sorter", ascending=False).head(50)

,pad-pre,snapshot-1,snapshot-2,snapshot-3,pad-past,click-pad-pre,click-snapshot-1,click-snapshot-2,click-snapshot-3,click-pad-past,slope,residuals,r_squared,sorter,temporal-type,rank
query_id,,,,,,,,,,,,,,,,
5396239,18,12,86,464,120,23,92,45,6443,136,0.171840,0.154215,0.656922,4.310301e-01,loosing,1
7787520,6,6,84,163,65,69,30,46,2201,89,0.173413,0.186303,0.617465,3.595144e-01,loosing,2
3481141,7,30,25,56,15,0,37,69,417,16,0.139709,0.114963,0.629332,2.390932e-01,loosing,3
4469163,0,5,5,5,1,0,32,46,82,0,0.110163,0.048638,0.713889,1.735539e-01,loosing,4
5009948,2,2,16,19,1,135,325,98,35,0,0.081344,0.017185,0.793831,1.718833e-01,loosing,5
16073517,0,4,3,3,1,0,43,41,86,0,0.126026,0.102262,0.608323,1.711919e-01,loosing,6
9067341,1,4,4,6,1,0,50,40,88,0,0.105061,0.043583,0.716923,1.696865e-01,loosing,7
4293493,2,5,9,9,1,339,67,40,217,0,0.077686,0.009438,0.864766,1.687490e-01,loosing,8
4223361,1,4,3,5,1,0,37,43,168,0,0.108503,0.064298,0.646764,1.681545e-01,loosing,9
